In [0]:
from pyspark.sql.functions import (
    current_timestamp,
    lit,
    sha2,
    concat_ws,
    to_date,
    monotonically_increasing_id
)

import uuid
import math

dbutils.widgets.text("source_table", "")
dbutils.widgets.text("target_table", "")
dbutils.widgets.text("jdbc_url", "")
dbutils.widgets.text("user", "")
dbutils.widgets.text("password", "")
dbutils.widgets.text("driver", "")
dbutils.widgets.text("source_name", "")
source_table = dbutils.widgets.get("source_table")
target_table = dbutils.widgets.get("target_table")

jdbc_url=dbutils.widgets.get("jdbc_url")
user=dbutils.widgets.get("user")
password=dbutils.widgets.get("password")
driver=dbutils.widgets.get("driver")
source_name = dbutils.widgets.get("source_name")


properties = {
    "user": user,
    "password": password,
    "driver": driver
}

df = spark.read.jdbc(
    url=jdbc_url,
    table=source_table,
    properties=properties
)


# =========================================================
# METADATA GENERATION
# =========================================================
lower_bound = 1
upper_bound = df.count()
batch_id = f"{lower_bound}_{upper_bound}"

pipeline_run_id = str(uuid.uuid4())

# Keep only original business columns for hash generation
business_columns = df.columns

df = (
    df.withColumn("migrated_at", current_timestamp())
      .withColumn("ingestion_date", to_date(current_timestamp()))
      .withColumn("batch_id", lit(batch_id))
      .withColumn("source_table", lit(source_table))
      .withColumn("source_system", lit(source_name))
      .withColumn("load_start_id", lit(lower_bound))
      .withColumn("load_end_id", lit(upper_bound))
      .withColumn("pipeline_run_id", lit(pipeline_run_id))
      .withColumn(
          "uniq_id",
          monotonically_increasing_id()
      )
)

df.write.format("delta") \
    .mode("append") \
    .saveAsTable(target_table)

print("SINGLE LOAD SUCCESS")